# Solution A2 · Random forest from scratch: the correlation between trees and the variance of the average

**Type**: Implement from scratch

## Objective

Build a forest by hand (trees with `max_features`) and measure, for `max_features` = all, `sqrt` and 2, the **mean correlation between the tree predictions**, the variance of a single tree and of the average, and the accuracy. Check the formula $\rho\sigma^2 + (1-\rho)\sigma^2/B$ and understand
why a forest of **worse** trees (with limited features) does better than bagging.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_classification(2000, n_features=12, n_informative=5, n_redundant=2, class_sep=0.8, flip_y=0.08, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0)
B = 60
options = [None, "sqrt", 2]

## Commented solution

In [ ]:
def forest(X, y, B, max_features, seed=0):
    rng = np.random.default_rng(seed)
    trees = []
    for b in range(B):
        i = rng.integers(0, len(X), len(X))
        trees.append(DecisionTreeClassifier(max_features=max_features, random_state=b).fit(X[i], y[i]))
    return trees

In [ ]:
rows = {}
for mf in options:
    trees = forest(X_tr, y_tr, B, mf)
    P = np.array([t.predict_proba(X_te)[:, 1] for t in trees])
    rho = (np.corrcoef(P).sum() - len(P)) / (len(P) * (len(P) - 1))                     # mean correlation between different pairs of trees
    single_var = P.var(axis=0).mean()                                                    # variance (over the trees) of a prediction, averaged over the examples
    rows[str(mf)] = {"rho": rho, "single_acc": ((P > 0.5) == y_te).mean(axis=1).mean(), "forest_acc": ((P.mean(0) > 0.5) == y_te).mean(),
                     "single_var": single_var, "mean_var_formula": rho * single_var + (1 - rho) * single_var / B}
table = pd.DataFrame(rows).T
table.round(4)

In [ ]:
best = table["forest_acc"].idxmax()
residual_variance_rho = table.loc[best, "rho"] * table.loc[best, "single_var"]
print(f"best: max_features={best} | variance that does not drop when adding trees (rho * sigma^2): {residual_variance_rho:.4f} | variance of a tree: {table.loc[best, 'single_var']:.4f}")

## Check

In [ ]:
with verify():
    check(list(table.columns) == ["rho", "single_acc", "forest_acc", "single_var", "mean_var_formula"] and list(table.index) == ["None", "sqrt", "2"], "table with three rows and five columns", "columns: rho, single_acc, forest_acc, single_var, mean_var_formula")
    check(table["rho"].is_monotonic_decreasing and table.loc["None", "rho"] > 0.45 and table.loc["2", "rho"] < 0.42, "the correlation between trees drops with max_features: from over 0.45 (all) to under 0.42 (2)", "mean of the off-diagonal coefficients of np.corrcoef(P)")
    check(table["single_acc"].is_monotonic_decreasing and table.loc["None", "single_acc"] - table.loc["2", "single_acc"] > 0.03, "a single tree with fewer features is worse (by over 3 points with max_features=2)", "mean accuracy of a tree")
    check(table.loc["sqrt", "forest_acc"] > table.loc["None", "forest_acc"] + 0.008 and table.loc["2", "forest_acc"] > table.loc["None", "forest_acc"], "but the forest with sqrt and with 2 features is better than bagging (max_features=all)", "forest_acc")
    check(best in ("sqrt", "2") and close(residual_variance_rho, table.loc[best, "rho"] * table.loc[best, "single_var"]), "the best is sqrt or 2; residual variance = rho * single variance", "the term that does not drop when adding trees")
    check((table["mean_var_formula"] < 0.6 * table["single_var"]).all() and table["mean_var_formula"].is_monotonic_decreasing, "the variance of the average is less than 60% of that of a tree (almost as much as rho), and drops with a smaller max_features", "rho * s2 + (1 - rho) * s2 / B")
done()

## What you learned

- **More trees are not enough when they are correlated.** The variance of the average of `B` trees is $\rho\sigma^2 + (1-\rho)\sigma^2/B$: adding trees eliminates only the second term. With all the features (bagging) $\rho = 0.53$: the variance will not drop below 53% of that of a tree even with infinite trees.
- **A small `max_features` reduces $\rho$** (0.53 → 0.43 with `sqrt` → 0.40 with 2 features): the trees do not all choose the dominant feature at the root, and so they make different errors.
- **The price is a worse tree:** the accuracy of a single tree drops (0.77 → 0.75 → 0.74), but the average gains more than each tree loses: 0.843 (bagging) against 0.865 (`sqrt`). It is the bias-variance trade-off in a surprising form: **weaker trees, stronger ensemble.**
- With a `max_features` that is too small the gain ends (the trees become too weak): that is why it is a hyperparameter to choose, on a small grid (`sqrt`, `log2`, fractions, all).

## Common pitfall

Choosing `max_features` by looking at the accuracy of **one tree**: you would only see its negative side. You always evaluate the whole forest (OOB or cross-validation).

## Interview question

<details><summary>Why does a random forest work better than bagging?</summary>

Because at every split it limits the candidate features: the trees end up less correlated (they do not all choose the same dominant feature) and the variance of the average is rho·σ² + (1−rho)·σ²/B, where extra trees alone do not reduce the correlation term. Each tree is a little worse (more bias),
but the average of less correlated trees gains much more. It is the trade-off that `max_features` controls.

</details>